# Pretrain — единый стриминговый микс датасетов

Собирает несколько корпусов **image → HTML** в **один поток** для continual pretraining
**без полного скачивания**: каждый источник грузится `streaming=True`, нормализуется к общей
схеме `{image, html, source}`, потоки смешиваются `interleave_datasets` с весами. На выходе —
`IterableDataset`, который отдаётся тренеру напрямую.

**Как это устроено (почему так):**
- `streaming=True` — данные качаются шардами по мере итерации, не целиком в память/на диск.
- `.map(normalize)` — приводим разные поля источников (`text` / `html`+`css` / …) к `{image, html, source}`.
- `interleave_datasets(probabilities=…)` — на каждом шаге сэмпл берётся из источника по вероятности;
  `stopping_strategy="all_exhausted"` — не останавливаться, пока не исчерпаны все.
- `.shuffle(buffer_size)` — потоковый шафл через буфер (полного перемешивания без длины нет).

Претрейн-формат намеренно **сырой** (`image` + `html`), без контрактной task-схемы (та — для SFT).
Форматирование во вход VLM — на стороне трен-кода.

## 1. Конфиг источников

In [ ]:
# name    — метка источника (попадает в поле "source")
# path/split — HF-датасет
# html/image — имена полей в источнике
# weight  — вес в миксе (нормируется в вероятность)
# real    — реальные страницы (нужен де-блоб data-URI); синтетике не нужен
SOURCES = [
    {"name": "websight",  "path": "HuggingFaceM4/WebSight",       "split": "train",
     "html": "text", "image": "image", "weight": 0.4, "real": False},
    {"name": "webcode2m", "path": "xcodemind/webcode2m_purified", "split": "train",
     "html": "text", "image": "image", "weight": 0.4, "real": True},
    # Web2Code — instruction-формат, ПРОВЕРИТЬ точные поля перед включением:
    # {"name": "web2code", "path": "MBZUAI/Web2Code", "split": "train",
    #  "html": "text", "image": "image", "weight": 0.2, "real": True},
]

SEED           = 0
SHUFFLE_BUFFER = 10_000   # буфер потокового шафла (память <-> качество перемешивания)
MIN_HTML_CHARS = 200      # отсев пустых/битых страниц (дёшево, в потоке)
DEBLOB         = True     # вырезать base64 data-URI у real-источников (в потоке)

## 2. Импорты + нормализация

In [ ]:
import re
from datasets import load_dataset, interleave_datasets, Image as HFImage

# де-блоб: убрать длинные data-URI (base64/inline) — как в анализе WebUI
_DATA_URI_RE = re.compile(r"data:[A-Za-z0-9+/=;,._%-]{40,}")
def strip_data_uris(html):
    return _DATA_URI_RE.sub("data:,", html) if html else html

def make_normalize(src):
    hf, imf, name, real = src["html"], src["image"], src["name"], src["real"]
    def fn(ex):
        html = ex.get(hf) or ""
        if real and DEBLOB:
            html = strip_data_uris(html)
        return {"image": ex.get(imf), "html": html, "source": name}
    return fn

## 3. Сборка потоков (streaming, без скачивания)

In [ ]:
streams, weights = [], []
for src in SOURCES:
    ds = load_dataset(src["path"], split=src["split"], streaming=True)
    orig_cols = list(ds.features) if ds.features else None   # убрать исходные поля -> общая схема
    ds = ds.map(make_normalize(src), remove_columns=orig_cols)
    ds = ds.filter(lambda ex: ex["image"] is not None and len(ex["html"]) >= MIN_HTML_CHARS)
    # единый тип колонки image (важно для interleave — схемы должны совпасть)
    ds = ds.cast_column("image", HFImage())
    streams.append(ds)
    weights.append(src["weight"])

probs = [w / sum(weights) for w in weights]
print("источников:", len(streams))
print("вероятности:", {s["name"]: round(p, 3) for s, p in zip(SOURCES, probs)})

## 4. Микс + потоковый шафл

In [ ]:
mix = interleave_datasets(
    streams, probabilities=probs, seed=SEED,
    stopping_strategy="all_exhausted",   # идём пока не исчерпаны ВСЕ источники
)
mix = mix.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)
mix   # IterableDataset -> отдаётся тренеру напрямую

## 5. Проверка выдачи

In [ ]:
from collections import Counter
N_PEEK = 20
peek = list(mix.take(N_PEEK))
print("источники в первых", N_PEEK, ":", dict(Counter(r["source"] for r in peek)))
r0 = peek[0]
print("поля:", list(r0.keys()))
print("image size:", r0["image"].size if r0["image"] is not None else None)
print("html head:", r0["html"][:120].replace(chr(10), " "))
r0["image"]   # покажет картинку

## Как отдать тренеру

`mix` — `IterableDataset`, передаётся в `Trainer` / цикл обучения напрямую:
- у стриминга **нет длины** → задавать `max_steps` (не `num_train_epochs`);
- шафл — потоковый (`shuffle(buffer_size)`); чем больше буфер, тем лучше перемешивание (и память);
- **форматирование во вход VLM** (image-токены + HTML как таргет) — на стороне трен-кода,
  поля `image` + `html` (переиспользовать `formatting.py` SFT-трека);
- **веса микса** (`weight`) — баланс синтетика/реальные (сейчас 40/40; Web2Code выкл — проверить поля);
- де-блоб включён для `real`-источников; фильтр по токенам в потоке дорог — при нужде добавить
  `.filter` по `token_len.count_tokens` (медленно) или по символам.

**Проверить перед боевым прогоном:**
1. Точные поля **Web2Code** (instruction-формат) перед включением.
2. Доступность датасетов (гейтинг HF / `HF_TOKEN`).
3. Что `interleave_datasets` не спотыкается на `features` — здесь выровнено `remove_columns`
   + `cast_column("image", Image())`; если источник отдаёт не PIL, поправить `normalize`.
4. Баланс весов под реальный объём/качество источников.